# 🎙️ OmniVoice Fixed Notebook
### Setup, Server, and Tunneling

In [ ]:
%cd /content/
!rm -rf omnivoice-colab
!git clone https://github.com/hahyt6644-gif/omnivoice-colab.git
%cd omnivoice-colab

!rm -rf OmniVoice
!git clone https://github.com/k2-fsa/OmniVoice.git

# Install cloudflared
!curl -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/bin/cloudflared
!chmod +x /usr/bin/cloudflared

!apt-get update -y && apt-get install -y ffmpeg
!pip install -q fastapi uvicorn[standard] python-multipart httpx requests aiofiles nest-asyncio pydub soundfile librosa
!pip install -q -r colab.txt

from IPython.display import clear_output
clear_output()
print('✅ INSTALLATION COMPLETE')

In [ ]:
import subprocess, time, re, requests

# Cleanup any existing instances on the port
!fuser -k 7860/tcp

print('🚀 Starting OmniVoice server...')
server = subprocess.Popen(['python', 'app.py'])

# Wait for server
for i in range(30):
    try:
        r = requests.get('http://localhost:7860', timeout=2)
        print('✅ Server ready')
        break
    except:
        time.sleep(2)

print('🌍 Starting tunnel...')
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:7860'], 
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

url = None
start = time.time()
while time.time() - start < 60:
    line = tunnel.stdout.readline()
    if 'trycloudflare.com' in line:
        match = re.search(r'https://[a-zA-Z0-9.-]+\.trycloudflare\.com', line)
        if match:
            url = match.group(0)
            break

if url:
    print(f'\n✅ PUBLIC URL: {url}')
else:
    print('❌ Tunnel failed')

In [ ]:
import nest_asyncio
nest_asyncio.apply()
print('Stable runtime active.')